In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from ipywidgets import interact, FloatSlider, Dropdown
import ipywidgets as widgets

# Определение последовательности

**Последовательность** — это отображение из множества натуральных чисел в множество вещественных чисел:

$$
a_n : \mathbb{N} \to \mathbb{R}, \quad n \mapsto a_n
$$

То есть, каждому натуральному $n$ поставлено в соответствие число $a_n$. Последовательность обозначается как $(a_n)_{n=1}^{\infty}$.

---

## Определение предела последовательности (формулировка Вейерштрасса)

Число $a \in \mathbb{R}$ называется **пределом последовательности** $(a_n)$, если:

$$
\forall \varepsilon > 0\ \exists n_0 \in \mathbb{N}\ \text{такое, что}\ \forall n > n_0:\ |a_n - a| < \varepsilon.
$$

Обозначается:

$$
\lim_{n \to \infty} a_n = a
$$



In [ ]:
def sequence(n: np.ndarray) -> np.ndarray:
    return 1/n

In [ ]:
#@title plot
def show_convergence(N: int, sequence):
    n_values = np.arange(1, N + 1)
    a_n_values = sequence(n_values)

    plt.figure(figsize=(10, 2))
    plt.hlines(y=0, xmin=min(a_n_values) - 0.1, xmax=max(a_n_values) + 0.1, color='lightgray')

    cmap = plt.cm.viridis
    norm = plt.Normalize(vmin=n_values.min(), vmax=n_values.max())
    colors = cmap(norm(n_values))

    plt.scatter(a_n_values, np.zeros_like(a_n_values), color=colors, s=50, label=r'$a_n$')

    plt.title('Convergence of $a_n$ on the Number Line')
    plt.xlabel('Real number line')
    plt.yticks([])
    plt.legend()
    plt.grid(True, axis='x')
    plt.show()



In [ ]:
show_convergence(100, sequence)

# Определение предела функции (в точке $x_0$)

Пусть $f: \mathbb{R} \to \mathbb{R}$ и $x_0 \in \mathbb{R}$. Тогда число $a \in \mathbb{R}$ называется **пределом функции** $f$ в точке $x_0$, если:

$$
\forall \varepsilon > 0\ \exists \delta > 0\ \text{такое, что}\ \forall x \in \mathbb{R},\ 0 < |x - x_0| < \delta:\ |f(x) - a| < \varepsilon.
$$




### Определение словами:

Число $a \in \mathbb{R}$ называется **пределом функции** $f$ в точке $x_0$, если:

> для **любого** положительного числа $\varepsilon > 0$ **существует** положительное число $\delta > 0$, такое что **для всех** $x \in \mathbb{R}$, удовлетворяющих неравенству $0 < |x - x_0| < \delta$, выполняется неравенство $|f(x) - a| < \varepsilon$.

Иными словами, значения функции $f(x)$, соответствующие аргументам $x$, лежащим в проколотой δ-окрестности точки $x_0$, находятся в пределах ε-окрестности числа $a$.





Обозначается:

$$
\lim_{x \to x_0} f(x) = a
$$


## Пример для $y = x^2$

In [ ]:
#@title plot
def f(x):
    return x**2

def compute_epsilon(x0, delta):
    x_test = np.linspace(x0 - delta, x0 + delta, 1000)
    x_test = x_test[x_test != x0]
    y_test = f(x_test)
    a = f(x0)
    return np.max(np.abs(y_test - a))

def plot_limit(x0=1.0, delta=0.5):
    a = f(x0)
    epsilon = compute_epsilon(x0, delta)
    x_vals = np.linspace(x0 - 4, x0 + 4, 1000)
    y_vals = f(x_vals)
    epsilon_mask = (np.abs(y_vals - a) < epsilon)
    delta_mask = (np.abs(x_vals - x0) < delta)
    both_mask = epsilon_mask & delta_mask

    plt.figure(figsize=(8, 6))
    plt.plot(x_vals, y_vals, label=r'$f(x) = x^2$', color='blue')
    plt.plot(x_vals[both_mask], y_vals[both_mask], color='orange', linewidth=3, label='Within ε-δ zone')
    plt.hlines([a - epsilon, a + epsilon], x_vals[0], x_vals[-1], colors='red', linestyles='--', label=r'$\varepsilon$-neighborhood')
    plt.fill_between(x_vals, a - epsilon, a + epsilon, color='red', alpha=0.1)
    plt.vlines([x0 - delta, x0 + delta], min(y_vals), max(y_vals), colors='green', linestyles='--', label=r'$\delta$-neighborhood')
    plt.fill_betweenx(np.linspace(min(y_vals), max(y_vals), 500), x0 - delta, x0 + delta, color='green', alpha=0.1)
    plt.plot(x0, a, 'ro', markerfacecolor='white', markersize=8, label='Limit point (open circle)')
    plt.title(f'Limit of $f(x) = x^2$ at $x_0 = {x0:.2f}$ with δ = {delta:.2f} → ε ≈ {epsilon:.3f}')
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.legend()
    plt.grid(True)
    plt.show()

interact(plot_limit,
         x0=FloatSlider(value=1.0, min=-2.0, max=7.0, step=0.1, description='x₀'),
         delta=FloatSlider(value=0.5, min=0.01, max=1.0, step=0.01, description='δ'))


# Derivative

Общее определение производной:

$$\frac{dy}{dx} = \lim\limits_{\Delta x \to 0} \frac{f(x+\Delta x) - f(x)}{\Delta x}$$

В этой работе мы научимся вычислять приближенные значения производных в точке:

$$\frac{dy}{dx}(x) \approx \frac{f(x + \varepsilon) - f(x)}{\varepsilon}$$

Где $\varepsilon$ - очень маленькое число.

## Task
Напишем программу рассчета значения производной в точке для функции

$$y = x^3 - 2x^2 + 6$$

In [ ]:
def func(x:float)->float:
  return (x**3 - 2*x**2 + 6)

def derivative(x:float, func)->float:
  'Derivative at point `x` of function `func`'
  dx = 10**(-6)
  return (func(x + dx) - func(x)) / dx

In [ ]:
func(4) # Значение функции f(x) при x=4

In [ ]:
derivative(4, func) # Значение производной функции f`(x) при x=4

Сделаем проверку:

$ y = x^3 - 2x^2 + 6 $

$ \frac{dy}{dx} = 3x^2 - 4x $

$ \frac{dy}{dx}(4) = 3*4^2 - 4*4 $

In [ ]:
3*4**2 - 4*4

Вывод: Наш код позволил вычислить значение производной с высокой точностью.

## Task

* Напишите программу для нахождения значений функции (Не забудьте про ограничения для аргумента: $x^2+2x+3 > 0$)

$$f(x) = \ln{\sqrt{x^2 + 2x + 3}}$$

* Постройте график производной! Используйте наш метод `derivative(x:float, func)`

* Что можно сказать про родительскую функцию оценивая график ее производной?

In [ ]:
def func_2(x):
  return np.log(np.sqrt(x**2 + 2*x +3))

func_2 = np.vectorize(func_2)

In [ ]:
xx = np.linspace(-5, 4, 50)
yy = func_2(xx)
yy_der = derivative(xx, func_2)
sns.lineplot(x=xx, y=yy, color='black', label='$y=f(x)$')
sns.lineplot(x=xx, y=yy_der, color='green', label='$y=\\frac{df}{dx}$')
plt.grid()

$$f(x) = \ln{\sqrt{x^2 + 2x + 3}}$$

$$\frac{df}{dx} = \frac{1}{\sqrt{x^2 + 2x + 3}} * \frac{1}{2\sqrt{x^2 + 2x + 3}} * (2x+2) = \frac{x+1}{x^2 + 2x + 3}$$

In [ ]:
xx = np.linspace(-6, 6, 100)
yy_der = derivative(xx, func_2)
yy_der_anal = (xx+1) / (xx**2 + xx*2 + 3)

sns.lineplot(x=xx, y=yy_der, color='black', label='auto')
sns.lineplot(x=xx, y=yy_der_anal, color='green', label='manual')
plt.grid()

# Касательная

Получим множество точек для построения  касательной к графику в точке $x_0$ и сделаем визуализацию.

$$y_{tan} = f'(x_0)(x-x_0) + f(x_0)$$

Пример реализации на Python:

```python
def tangent(x0, func):
    dx = 1e-6
    k = (func(x0 + dx) - func(x0)) / dx
    xxx = np.linspace(x0 - 5, x0 + 5, 100)
    yyy = (xxx - x0) * k + func(x0)
    return xxx, yyy

```



---

Напишем пример для функции $y = \sin{x}$ .

Далее вы можете опробовать какие-либо другие зависимости))

In [ ]:
def func(x:float)->float:
  return np.sin(x)

def tangent(x0, func):
  dx = 1e-6
  k = (func(x0 + dx) - func(x0)) / dx
  xxx = np.linspace(x0 - 5, x0 + 5, 100)
  yyy = (xxx - x0) * k + func(x0)
  return xxx, yyy

In [ ]:
#@title plot
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, FloatSlider

func_np = np.vectorize(func)

def tangent(x0, func):
    dx = 1e-6
    k = (func(x0 + dx) - func(x0)) / dx
    xxx = np.linspace(x0 - 5, x0 + 5, 100)
    yyy = (xxx - x0) * k + func(x0)
    return xxx, yyy

def plot_with_tangent(x0):
    xx = np.linspace(x0 - 5, x0 + 5, 400)
    yy = func_np(xx)
    xxx, yyy = tangent(x0, func)

    plt.figure(figsize=(10, 6))
    sns.lineplot(x=xx, y=yy, color='g', label='Function')
    sns.lineplot(x=xxx, y=yyy, color='r', label='Tangent')
    plt.plot(x0, func(x0), 'ko', markersize=8, label='Touch point')
    plt.title(f'Tangent to f(x) at x₀ = {x0:.2f}')
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.grid(True)
    plt.xlim(x0 - 5, x0 + 5)
    plt.ylim(min(yy) - 1, max(yy) + 1)
    plt.legend()
    plt.show()

interact(plot_with_tangent, x0=FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='x₀'))


# Decorator *

Данный раздел не обязательный и подходит для итстинных ценителей абстрактного трэша! ОСТОРОЖНО!

Я предупреждал...
###  Что такое **декоратор функции** в Python?

**Декоратор** — это специальная конструкция, которая позволяет **модифицировать поведение функции** без изменения её исходного кода.

Декоратор — это **функция, принимающая другую функцию как аргумент**, и возвращающая **новую функцию** с изменённым поведением. Декорирование - это "оборачивание" какой-то функции некоторой "оберткой" так, что ваша функция кроме того что она делает сама по себе, начинает обладать каким-то дополнительным функционалом!



## Пример

In [ ]:
def cat()->None:
  print('Мяу....')

cat()

In [ ]:
def barsik(func):
  def wrapper():
    func()

    print('\nКот делает КУСЬ и убегает.......')

  return wrapper

In [ ]:
@barsik
def cat():
  print('Мяу....')

cat()

## Как это работает с производными

```python
@autodiff
def func(x):
    ...
```

Это **эквивалентно** следующему:

```python
def func(x):
    ...

func = autodiff(func)
```

То есть, сначала создаётся функция `func`, потом она "оборачивается" в декоратор `autodiff`, и уже обёрнутая версия под тем же именем становится доступной.

---

### Что делает `@autodiff` в этом примере?

* При вызове `func(x)` теперь фактически вызывается **производная от этой функции**, численно приближённая по формуле:

  $$
  f'(x) \approx \frac{f(x + dx) - f(x)}{dx}
  $$
* Это реализовано с помощью вложенной функции `wrapper`.


In [ ]:
def autodiff(func):

  def wrapper(x):
    print('___AUTO DIFF ___')
    dx = 10**(-6)
    return ( func(x + dx) - func(x) ) / dx
  return wrapper


In [ ]:
@autodiff
def func(x:float)->float:
  return np.sin(x) - np.cos(x)

In [ ]:
func(3)

# Нахождение минимума функции

Когда мы находим локальные максимумы и минимумы функций руками, мы применяем аналитический подход:

$$\frac{\mathrm{d}f}{\mathrm{d}x} = 0$$
Решая данное уравнение мы находим критические точки, среди которых могут быть точки экстремумов.

Однако в программной среде приняты другие алгоритмы, которые итеративным методом позволяют нам найти экстремумы.

Мы предложим подход, который положен в основу "Градиентного спука" с адаптацией для функции одного аргумента $f: \mathbb{R} \to \mathbb{R}$, который нам поможет найти локальный минимум функции:

$$x^n = x^{n-1} - \eta \frac{\mathrm{d}f}{\mathrm{d}x}(x^{n-1})$$


**где:**

* $x^{n}$ — значение аргумента на **n-м шаге** алгоритма;
* $x^{n-1}$ — значение аргумента на **предыдущем (n−1) шаге**;
* $\eta$ (эта, «learning rate») — **скорость обучения**, то есть шаг, с которым мы двигаемся в сторону минимума;
* $\frac{\mathrm{d}f}{\mathrm{d}x}(x^{n-1})$ — **производная функции** $f$ по переменной $x$, вычисленный в точке $x^{n-1}$.. (позже для многомерных случаев мы это будем называть градиентом функции);

Важной особенностью является необходимость инициализировать алгоритм - то есть выбрать произвольную точку с которой у нас начнется поиск локального минимума!




---
Найдем локальный минимум функции:

$$f(x) = \ln{\sqrt{x^2 + 2x + 3}}$$



In [ ]:
def func(x:float)->float:
  return np.log((x**2 + 2*x + 3)**.5)

def grad(fun, x:float)->float:
  dx = 10**(-6)
  return (fun(x + dx) - fun(x)) / dx

In [ ]:
current_x = -4

N_ITER = 20

LearningRate = 2

history = []
grad_norm = []

for _ in range(N_ITER):
  history.append(current_x)
  gradient = grad(func, current_x)
  grad_norm.append(gradient)
  current_x = current_x - LearningRate * gradient

history.append(current_x)

In [ ]:
history[-1]

In [ ]:
xx=np.linspace(-5, 5, 100)
yy = func(xx)
history_x = np.array(history)
history_y = func(history_x)

sns.lineplot(x=xx, y=yy, color='black', label='$f(x) = \ln{\sqrt{x^2 + 2x + 3}}$')
sns.scatterplot(x=history_x, y=history_y, color='r', s=70, alpha=.6, edgecolor='black', label='Gradient steps')
plt.plot(history_x, history_y, color='g')
plt.title(f'Optimum: $x = {history_x[-1].round(2).item()}$')
plt.grid()
plt.legend()

In [ ]:
sns.lineplot(grad_norm)
plt.grid()

## Stop Critery

In [ ]:
current_x = -2
MAX_ITER = 1000
LearningRate = .6
stop_critery = 1e-6
history = []
grad_norm = []

counter = 0

for _ in range(MAX_ITER):
    counter += 1
    history.append(current_x)
    gradient = grad(func, current_x)
    grad_norm.append(gradient)
    delta = LearningRate * gradient
    if abs(delta) < stop_critery:
        print(f'DELTA: {delta}')
        current_x = current_x - delta
        break
    current_x = current_x - delta


history.append(current_x)
print(f'LOCAL MIN: x = {current_x}')
print(f'NUM ITERATION: {counter}')

In [ ]:
history

In [ ]:
grad_norm

In [ ]:
xx=np.linspace(-5, 5, 100)
yy = func(xx)
history_x = np.array(history)
history_y = func(history_x)

sns.lineplot(x=xx, y=yy, color='black', label='$f(x) = \ln{\sqrt{x^2 + 2x + 3}}$')
sns.scatterplot(x=history_x, y=history_y, color='r', s=70, alpha=.6, edgecolor='black', label='Gradient steps')
plt.plot(history_x, history_y, color='g')
plt.title(f'Optimum: $x = {history_x[-1].round(2).item()}$')
plt.grid()
plt.legend()

In [ ]:
plt.subplots(figsize=(10, 2))
plt.scatter(history_x, np.zeros_like(history_x), color='g', s=70, alpha=.7, edgecolor='black')
plt.grid()
plt.yticks([])
plt.xlabel('Argument')
plt.title('Gradient steps')

## Task

Найдите локальный минимум функции методом, описанным выше:

$$y = x^4 -3x^3 +3x +1$$

Отследите с помощью визуализации, насколько корректно отработал алгоритм. С какими проблемами вы столкнулись?

# Производная функции нескольких переменных

Рассмотрим пример:

$$f(x, y) = A * e^{-(\frac{x^2}{2\sigma^2_x} + \frac{y^2}{2\sigma_y^2})}$$

In [ ]:
A = 0.3
sigma_x = 0.8
sigma_y = 1.0


def gaussian_2d(x, y, A=A, sigma_x=sigma_x, sigma_y=sigma_y):
    """
    Computes the value of a 2D Gaussian at point (x, y).

    Works with both scalars and numpy arrays
    (thanks to numpy vectorization).

    Parameters
    ----------
    x, y : float or np.ndarray
        Coordinates at which the function is evaluated.
    A : float
        Amplitude (peak height).
    sigma_x, sigma_y : float
        Standard deviations along the x and y axes.

    Returns
    -------
    float or np.ndarray
        The value of f(x, y).
    """
    return A * np.exp(-((x**2) / (2 * sigma_x**2) + (y**2) / (2 * sigma_y**2)))




In [ ]:
#@title plot
def plot_gaussian_surface(x_range=(-2.5, 2.5), y_range=(-2.5, 2.5), n_points=200):
    """
    Plots the 2D Gaussian: 3D surface on the left, contour lines on the right.
    """
    x = np.linspace(*x_range, n_points)
    y = np.linspace(*y_range, n_points)
    X, Y = np.meshgrid(x, y)
    Z = gaussian_2d(X, Y)

    fig = plt.figure(figsize=(14, 6))

    # Left: 3D wireframe surface
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    ax1.plot_wireframe(X, Y, Z, color='black', alpha=0.4, rstride=10, cstride=10)
    ax1.set_title("2D Gaussian Surface")
    ax1.set_xlabel("x")
    ax1.set_ylabel("y")
    ax1.set_zlabel("f(x, y)")

    # Right: contour lines in the xOy plane (step = 0.01)
    ax2 = fig.add_subplot(1, 2, 2)
    step = 0.01
    levels = np.arange(step, Z.max() + step, step)
    contours = ax2.contour(X, Y, Z, levels=levels, colors='black', linewidths=1)
    ax2.clabel(contours, inline=True, fontsize=8, fmt="%.2f")
    ax2.set_title("Contour Lines (xOy plane)")
    ax2.set_xlabel("x")
    ax2.set_ylabel("y")
    ax2.set_aspect('equal')
    ax2.grid(True, linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_gaussian_surface()

Линия уровня (или множество уровня) функции $f: \mathbb{R}^n \to \mathbb{R}$
 — это множество точек области определения, в которых функция принимает заданное постоянное значение:

$L_c = \{ \mathbf{x} \in \mathbb{R}^n : f(\mathbf{x}) = c \}$

Линии уровня функции f(x, y) представляют собой проекции на плоскость xOy кривых, вдоль которых функция сохраняет постоянное значение. Расстояние между соседними линиями уровня, построенными с фиксированным шагом по z, обратно пропорционально модулю градиента функции в данной области: чем гуще расположены линии, тем быстрее изменяется f(x, y), то есть тем больше длина вектора градиента ∇f и круче наклон поверхности. И наоборот, разрежённые линии уровня соответствуют пологим участкам, где частные производные ∂f/∂x и ∂f/∂y малы по абсолютной величине. Таким образом, картина линий уровня даёт наглядное геометрическое представление о скорости роста функции и направлении её наибольшего возрастания, перпендикулярном линиям уровня в каждой точке.

## **Частная производная.**

Пусть задана функция $f: \mathbb{R}^n \to \mathbb{R}$, где $\mathbf{x} \in \mathbb{R}^n$ и $\mathbf{x} = (x_1, x_2, \ldots, x_n)$. Частной производной функции $f$ по переменной $x_i$ в точке $\mathbf{x}$ называется предел

$$
\frac{\partial f}{\partial x_i}(\mathbf{x}) = \lim_{h \to 0} \frac{f(x_1, \ldots, x_{i-1},\, x_i + h,\, x_{i+1}, \ldots, x_n) - f(x_1, x_2, \ldots, x_n)}{h},
$$

если этот предел существует и конечен. Эквивалентно, обозначая через $\mathbf{e}_i$ $i$-й орт стандартного базиса в $\mathbb{R}^n$, можно записать

$$
\frac{\partial f}{\partial x_i}(\mathbf{x}) = \lim_{h \to 0} \frac{f(\mathbf{x} + h\,\mathbf{e}_i) - f(\mathbf{x})}{h}.
$$

**Градиент.** Если все частные производные функции $f$ существуют в точке $\mathbf{x}$, то градиентом функции $f$ в этой точке называется вектор

$$
\nabla f(\mathbf{x}) = \left( \frac{\partial f}{\partial x_1}(\mathbf{x}),\; \frac{\partial f}{\partial x_2}(\mathbf{x}),\; \ldots,\; \frac{\partial f}{\partial x_n}(\mathbf{x}) \right) \in \mathbb{R}^n.
$$

Геометрически градиент $\nabla f(\mathbf{x})$ указывает направление наибыстрейшего возрастания функции в точке $\mathbf{x}$, а его модуль $\|\nabla f(\mathbf{x})\|$ равен скорости этого возрастания.

In [ ]:
#@title plot
A = 0.3
sigma_x = 0.8
sigma_y = 1.0


def gaussian_2d(x, y, A=A, sigma_x=sigma_x, sigma_y=sigma_y):
    """2D Gaussian: works with scalars or numpy arrays."""
    return A * np.exp(-((x**2) / (2 * sigma_x**2) + (y**2) / (2 * sigma_y**2)))


def partial_derivative(x0, y0, axis, h=1e-6):
    """
    Numerical partial derivative of f(x, y) at point (x0, y0).
    axis = 'x'  ->  df/dx
    axis = 'y'  ->  df/dy
    """
    if axis == 'x':
        return (gaussian_2d(x0 + h, y0) - gaussian_2d(x0, y0)) / h
    else:
        return (gaussian_2d(x0, y0 + h) - gaussian_2d(x0, y0)) / h

def plot_partial_dashboard(axis='x', x0=0.0, y0=0.0):
    """
    Interactive dashboard for visualizing partial derivatives.

    Left  : 2D cross-section of the surface along the chosen axis,
            with the tangent line at the current point.
    Right : contour map in the xOy plane with a red dot at (x0, y0)
            and the trace of the cutting plane (dashed green line).
    """
    # Grid for the contour plot
    x_grid = np.linspace(-2, 2, 200)
    y_grid = np.linspace(-2.5, 2.5, 200)
    X, Y = np.meshgrid(x_grid, y_grid)
    Z = gaussian_2d(X, Y)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # ---------- Left: cross-section + tangent ----------
    if axis == 'x':
        # df/dx: y is fixed at y0, vary x
        t = np.linspace(-2, 2, 300)
        section = gaussian_2d(t, y0)
        f0 = gaussian_2d(x0, y0)
        k = partial_derivative(x0, y0, 'x')
        tangent_x = np.linspace(x0 - 1.0, x0 + 1.0, 50)
        tangent_y = k * (tangent_x - x0) + f0

        ax1.plot(t, section, color='black', linewidth=2,
                 label=f'f(x, y={y0:.2f})')
        ax1.plot(tangent_x, tangent_y, color='green', linewidth=2,
                 label='tangent')
        ax1.plot(x0, f0, 'ro', markersize=8)
        ax1.set_xlabel("x")
        ax1.set_title(f"∂f/∂x at ({x0:.2f}, {y0:.2f}) ≈ {k:.2f}")
    else:
        # df/dy: x is fixed at x0, vary y
        t = np.linspace(-2.5, 2.5, 300)
        section = gaussian_2d(x0, t)
        f0 = gaussian_2d(x0, y0)
        k = partial_derivative(x0, y0, 'y')
        tangent_x = np.linspace(y0 - 1.0, y0 + 1.0, 50)
        tangent_y = k * (tangent_x - y0) + f0

        ax1.plot(t, section, color='black', linewidth=2,
                 label=f'f(x={x0:.2f}, y)')
        ax1.plot(tangent_x, tangent_y, color='green', linewidth=2,
                 label='tangent')
        ax1.plot(y0, f0, 'ro', markersize=8)
        ax1.set_xlabel("y")
        ax1.set_title(f"∂f/∂y at ({x0:.2f}, {y0:.2f}) ≈ {k:.2f}")

    # Fixed axes for the cross-section plot
    ax1.set_xlim(-2.5, 2.5)
    ax1.set_ylim(-0.05, A + 0.05)
    ax1.set_ylabel("f")
    ax1.grid(True, linestyle='--', alpha=0.4)
    ax1.legend(loc='upper right')

    # ---------- Right: contour map + current point + cutting plane trace ----------
    step = 0.01
    levels = np.arange(step, Z.max() + step, step)
    contours = ax2.contour(X, Y, Z, levels=levels, colors='black', linewidths=1)
    ax2.clabel(contours, inline=True, fontsize=7, fmt="%.2f")

    # Trace of the cutting plane in the xOy plane
    if axis == 'x':
        # Cutting plane is y = y0  ->  horizontal line on the contour map
        ax2.axhline(y=y0, color='green', linestyle='--', linewidth=1,
                    label=f'cutting plane y = {y0:.2f}')
    else:
        # Cutting plane is x = x0  ->  vertical line on the contour map
        ax2.axvline(x=x0, color='green', linestyle='--', linewidth=1,
                    label=f'cutting plane x = {x0:.2f}')

    ax2.plot(x0, y0, 'ro', markersize=10)
    ax2.set_title("Contour Lines (xOy plane)")
    ax2.set_xlabel("x")
    ax2.set_ylabel("y")
    ax2.set_aspect('equal')
    ax2.grid(True, linestyle='--', alpha=0.3)
    ax2.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.show()


interact(
    plot_partial_dashboard,
    axis=Dropdown(options=['x', 'y'], value='x', description='Axis'),
    x0=FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.05, description='x'),
    y0=FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.05, description='y'),
)

Построим градиенты данной фукции:

$$f(x, y) = A * e^{-(\frac{x^2}{2\sigma^2_x} + \frac{y^2}{2\sigma_y^2})}$$


$$
\nabla_{f(x, y)} =
\begin{bmatrix}
\frac{\partial f}{\partial x} \\
\frac{\partial f}{\partial y}
\end{bmatrix}
=
\begin{bmatrix}
\displaystyle -\frac{A x}{\sigma_x^2} \cdot e^{ -\left( \frac{x^2}{2\sigma_x^2} + \frac{y^2}{2\sigma_y^2} \right) } \\
\\
\displaystyle -\frac{A y}{\sigma_y^2} \cdot e^{ -\left( \frac{x^2}{2\sigma_x^2} + \frac{y^2}{2\sigma_y^2} \right) }
\end{bmatrix}
$$



In [ ]:
#@title plot
A = 0.3
sigma_x = 0.8
sigma_y = 1.0

x = np.linspace(-2, 2, 20)
y = np.linspace(-2, 2, 20)
X, Y = np.meshgrid(x, y)

Z = A * np.exp(-((X**2)/(2 * sigma_x**2) + (Y**2)/(2 * sigma_y**2)))

# ВАЖНО: сначала dy, потом dx
dZ_dy, dZ_dx = np.gradient(Z, y, x)

plt.figure(figsize=(8, 6))
plt.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.8)
plt.quiver(X, Y, dZ_dx, dZ_dy, color='white', scale=2.5, width=0.005)
plt.title("Gradient Field of 2D Gaussian")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)
plt.colorbar(label="f(x, y)")
plt.show()


In [ ]:
#@title plot
# Gaussian parameters
A = 0.3
sigma_x = 0.8
sigma_y = 1.0


def gaussian_2d(x, y, A=A, sigma_x=sigma_x, sigma_y=sigma_y):
    """2D Gaussian: works with scalars or numpy arrays."""
    return A * np.exp(-((x**2) / (2 * sigma_x**2) + (y**2) / (2 * sigma_y**2)))


def partial_derivative(x0, y0, axis, h=1e-6):
    """Numerical partial derivative of f at (x0, y0) along the given axis."""
    if axis == 'x':
        return (gaussian_2d(x0 + h, y0) - gaussian_2d(x0, y0)) / h
    else:
        return (gaussian_2d(x0, y0 + h) - gaussian_2d(x0, y0)) / h


def plot_gradient_dashboard(x0=0.0, y0=0.0):
    """
    Interactive dashboard for visualizing both partial derivatives
    and the gradient vector at point (x0, y0).

    Left  (top)    : cross-section f(x, y0) with tangent (illustrates df/dx)
    Left  (bottom) : cross-section f(x0, y) with tangent (illustrates df/dy)
    Right          : contour map with red gradient arrow at (x0, y0)
    """
    # Grid for the contour plot
    x_grid = np.linspace(-2, 2, 200)
    y_grid = np.linspace(-2.5, 2.5, 200)
    X, Y = np.meshgrid(x_grid, y_grid)
    Z = gaussian_2d(X, Y)

    # Compute values at the current point
    f0 = gaussian_2d(x0, y0)
    df_dx = partial_derivative(x0, y0, 'x')
    df_dy = partial_derivative(x0, y0, 'y')

    # Layout: 2 rows x 2 columns, right column spans both rows
    fig = plt.figure(figsize=(14, 7))
    gs = gridspec.GridSpec(2, 2, width_ratios=[1.2, 1], height_ratios=[1, 1],
                           wspace=0.25, hspace=0.35)
    ax_top = fig.add_subplot(gs[0, 0])
    ax_bot = fig.add_subplot(gs[1, 0])
    ax_cnt = fig.add_subplot(gs[:, 1])

    # ---------- Top-left: cross-section along x (df/dx) ----------
    t = np.linspace(-2, 2, 300)
    section_x = gaussian_2d(t, y0)
    tan_x = np.linspace(x0 - 1.0, x0 + 1.0, 50)
    tan_y = df_dx * (tan_x - x0) + f0

    ax_top.plot(t, section_x, color='black', linewidth=2,
                label=f'f(x, y={y0:.2f})')
    ax_top.plot(tan_x, tan_y, color='green', linewidth=2, label='tangent')
    ax_top.plot(x0, f0, 'ro', markersize=8)
    ax_top.set_xlim(-2.5, 2.5)
    ax_top.set_ylim(-0.05, A + 0.05)
    ax_top.set_xlabel("x")
    ax_top.set_ylabel("f")
    ax_top.set_title(f"∂f/∂x at ({x0:.2f}, {y0:.2f}) ≈ {df_dx:.2f}")
    ax_top.grid(True, linestyle='--', alpha=0.4)
    ax_top.legend(loc='upper right', fontsize=8)

    # ---------- Bottom-left: cross-section along y (df/dy) ----------
    t = np.linspace(-2.5, 2.5, 300)
    section_y = gaussian_2d(x0, t)
    tan_x = np.linspace(y0 - 1.0, y0 + 1.0, 50)
    tan_y = df_dy * (tan_x - y0) + f0

    ax_bot.plot(t, section_y, color='black', linewidth=2,
                label=f'f(x={x0:.2f}, y)')
    ax_bot.plot(tan_x, tan_y, color='green', linewidth=2, label='tangent')
    ax_bot.plot(y0, f0, 'ro', markersize=8)
    ax_bot.set_xlim(-2.5, 2.5)
    ax_bot.set_ylim(-0.05, A + 0.05)
    ax_bot.set_xlabel("y")
    ax_bot.set_ylabel("f")
    ax_bot.set_title(f"∂f/∂y at ({x0:.2f}, {y0:.2f}) ≈ {df_dy:.2f}")
    ax_bot.grid(True, linestyle='--', alpha=0.4)
    ax_bot.legend(loc='upper right', fontsize=8)

    # ---------- Right: contour map + gradient arrow ----------
    step = 0.01
    levels = np.arange(step, Z.max() + step, step)
    contours = ax_cnt.contour(X, Y, Z, levels=levels, colors='black', linewidths=1)
    ax_cnt.clabel(contours, inline=True, fontsize=7, fmt="%.2f")

    # Gradient arrow: scaled for visibility
    grad_norm = np.hypot(df_dx, df_dy)
    scale = 3.0  # visual scaling factor for the arrow
    if grad_norm > 1e-8:
        ax_cnt.arrow(x0, y0, df_dx * scale, df_dy * scale,
                     head_width=0.12, head_length=0.15,
                     fc='red', ec='red', length_includes_head=True,
                     linewidth=2, zorder=5)
    ax_cnt.plot(x0, y0, 'ro', markersize=8, zorder=6)

    ax_cnt.set_title(f"∇f ≈ ({df_dx:.2f}, {df_dy:.2f}),  |∇f| ≈ {grad_norm:.2f}")
    ax_cnt.set_xlabel("x")
    ax_cnt.set_ylabel("y")
    ax_cnt.set_aspect('equal')
    ax_cnt.grid(True, linestyle='--', alpha=0.3)

    plt.show()


interact(
    plot_gradient_dashboard,
    x0=FloatSlider(value=0.5, min=-2.0, max=2.0, step=0.05, description='x'),
    y0=FloatSlider(value=0.5, min=-2.0, max=2.0, step=0.05, description='y'),
)



---

Рассмотрим похожий пример но с локальным минимумом:

$$f(x, y) = -A * e^{-(\frac{x^2}{2\sigma^2_x} + \frac{y^2}{2\sigma_y^2})}$$

In [ ]:
#@title plot
A = -.3
sigma_x = 0.8
sigma_y = 1
x = np.linspace(-2, 2, 100)
y = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x, y)
Z = A * np.exp(-((X**2)/(2 * sigma_x**2) + (Y**2)/(2 * sigma_y**2)))

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

ax.plot_wireframe(X, Y, Z, color='black', alpha=0.5)

ax.set_title("2D Gaussian Surface")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("$f(x, y)$")

plt.tight_layout()
plt.show()

In [ ]:
#@title plot
A = -0.3
sigma_x = 0.8
sigma_y = 1.0

x = np.linspace(-2, 2, 20)
y = np.linspace(-2, 2, 20)
X, Y = np.meshgrid(x, y)

Z = A * np.exp(-((X**2)/(2 * sigma_x**2) + (Y**2)/(2 * sigma_y**2)))

# ВАЖНО: сначала dy, потом dx
dZ_dy, dZ_dx = np.gradient(Z, y, x)

plt.figure(figsize=(8, 6))
plt.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.8)
plt.quiver(X, Y, dZ_dx, dZ_dy, color='white', scale=2.5, width=0.005)
plt.title("Gradient Field of 2D Gaussian")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)
plt.colorbar(label="f(x, y)")
plt.show()

# Приближенное вычисление частных производных

$$\frac{∂f(x, y)}{∂x}(x, y) \approx \frac{f(x + \varepsilon, y) - f(x, y)}{\varepsilon}$$

$$\frac{∂f(x, y)}{∂y}(x, y) \approx \frac{f(x, y + \varepsilon) - f(x, y)}{\varepsilon}$$

Давайте снова поработаем с нашем Гаусcианом:

$$f(x, y) = A * e^{-(\frac{x^2}{2\sigma^2_x} + \frac{y^2}{2\sigma_y^2})}$$

Где:

$A = 0.3$

$\sigma_x = 0.8$

$\sigma_y = 1$

In [ ]:
def gaussian(x:float, y:float)->float:
  A = 0.3
  sigma_x = 0.8
  sigma_y = 1.0
  return A * np.exp(-((x**2)/(2 * sigma_x**2) + (y**2)/(2 * sigma_y**2)))

Методы приближенных вычислений частных производных и градиента:

In [ ]:
def df_dx(x:float, y:float, func)->float:
  dx = 10**(-6)
  return round((func(x+dx, y) - func(x, y)) / dx, 1)

def df_dy(x:float, y:float, func)->float:
  dy = 10**(-6)
  return round((func(x, y+dy) - func(x, y)) / dy, 1)

def gradient(x:float, y:float, func)->np.ndarray:
  return np.array([df_dx(x, y, func),
                   df_dy(x, y, func)])

In [ ]:
df_dx(1, -1, gaussian)

In [ ]:
df_dy(1, -1, gaussian)

In [ ]:
gradient(1, -1, gaussian)

# Реализация градиентного спуска

$$f(x, y) = -A * e^{-(\frac{x^2}{2\sigma^2_x} + \frac{y^2}{2\sigma_y^2})}$$

$$\{x_1, x_2, ..., x_n\}^k = \{x_1, x_2, ..., x_n\}^{k-1} - \eta\nabla_{f(x_1, x_2, ..., x_n)} $$


In [ ]:
def gaussian(x:float, y:float)->float:
  A = -0.3
  sigma_x = 0.8
  sigma_y = 1.0
  return A * np.exp(-((x**2)/(2 * sigma_x**2) + (y**2)/(2 * sigma_y**2)))

def df_dx(x:float, y:float, func)->float:
  dx = 10**(-6)
  return (func(x+dx, y) - func(x, y)) / dx

def df_dy(x:float, y:float, func)->float:
  dy = 10**(-6)
  return (func(x, y+dy) - func(x, y)) / dy

def gradient(x:float, y:float, func)->np.ndarray:
  return np.array([df_dx(x, y, func),
                   df_dy(x, y, func)])

In [ ]:
LEARNING_RATE = 1
N_ITER = 40
current_point = np.array([-1.8, .7])

history = [current_point]
for _ in range(N_ITER):
  current_point = current_point - LEARNING_RATE*gradient(x=current_point[0], y=current_point[1], func=gaussian)
  history.append(current_point)

print(*history, sep='\n')
history = np.array(history)

In [ ]:
#@title history plot
A = -0.3
sigma_x = 0.8
sigma_y = 1.0

x = np.linspace(-2, 2, 100)
y = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x, y)
Z = A * np.exp(-((X**2) / (2 * sigma_x**2) + (Y**2) / (2 * sigma_y**2)))

Z_history = A * np.exp(-((history[:, 0]**2) / (2 * sigma_x**2) +
                         (history[:, 1]**2) / (2 * sigma_y**2)))

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_wireframe(X, Y, Z, color='black', alpha=0.3)

ax.plot(history[:, 0], history[:, 1], Z_history, color='red', marker='o', label='Gradient Descent Path')

ax.set_title("2D Gaussian Surface with Gradient Descent Path")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("$f(x, y)$")
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
#@title plot
A = -0.3
sigma_x = 0.8
sigma_y = 1.0


x = np.linspace(-2, 2, 20)
y = np.linspace(-2, 2, 20)
X, Y = np.meshgrid(x, y)
Z = A * np.exp(-((X**2)/(2 * sigma_x**2) + (Y**2)/(2 * sigma_y**2)))


dZ_dy, dZ_dx = np.gradient(Z, y, x)

Z_history = A * np.exp(-((history[:, 0]**2)/(2 * sigma_x**2) + (history[:, 1]**2)/(2 * sigma_y**2)))

plt.figure(figsize=(8, 6))
plt.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.8)
plt.quiver(X, Y, dZ_dx, dZ_dy, color='black', scale=2.5, width=0.005)

plt.plot(history[:, 0], history[:, 1], color='red', linewidth=2)
plt.scatter(history[:, 0], history[:, 1], color='red', s=30)

plt.title("Gradient Field and Descent Path")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)
plt.colorbar(label="f(x, y)")
plt.show()


# Task

Методом градиентного спуска найдите минимум данной функции.

In [ ]:
def gauss_mixture(x, y)->float:
  A_1 = -0.3
  sigma_x_1 = 1.0
  sigma_y_1 = 1.2
  A_2 = -0.4
  sigma_x_2 = 1
  sigma_y_2 = 1.2
  z = A_1 * np.exp(-(((x-.5)**2)/(2 * sigma_x_1**2) + ((y-1)**2)/(2 * sigma_y_1**2))) + \
  A_2 * np.exp(-(((x + 1)**2)/(2 * sigma_x_2**2) + ((y + 2)**2)/(2 * sigma_y_2**2)))
  return z

In [ ]:
#@title plot
X, Y = np.meshgrid(np.linspace(-4, 4, 100), np.linspace(-6, 3, 100))
Z = gauss_mixture(X, Y)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

ax.plot_wireframe(X, Y, Z, color='black', alpha=0.5)

ax.set_title("2D Gaussian Surface")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("$f(x, y)$")

plt.tight_layout()
plt.show()

In [ ]:
X, Y = np.meshgrid(np.linspace(-4, 4, 100), np.linspace(-5, 3, 100))
Z = gauss_mixture(X, Y)

plt.figure(figsize=(8, 6))
plt.contourf(X, Y, Z, levels=40, cmap='magma', alpha=0.8)

plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)
plt.colorbar(label="f(x, y)")

In [ ]:
# Your code